##### Single Ingest

In [3]:
pip install pyroaring

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyroaring import BitMap

In [ ]:
# doc1 = {
#     "name": "Alex",
#     "theme": "dark",
#     "country": "USA"
# }

# doc2 = {
#     "name": "John",
#     "theme": "dark",
#     "country": "India"
# }

# doc3 = {
#     "name": "Alex",
#     "theme": "light",
#     "country": "India"
# }


# engine = SearchEngine()

# engine.index_document(doc1)
# engine.index_document(doc2)
# engine.index_document(doc3)

In [7]:
# doc4 = {
#     "name": "Pragya",
#     "theme": "light",
#     "country": "India"
# }
# engine.index_document(doc4)

In [8]:
# print(engine.get_cardinality())

In [9]:
# engine.disable_high_cardinality(2)

In [10]:
# doc5 = {
#     "name": "Rachna",
#     "theme": "light",
#     "country": "India"
# }
# engine.index_document(doc5)

In [11]:
#name not updated becoz name cardinality false
# print(engine.get_cardinality())

In [ ]:
# engine.lookup("name", "Alex")
# {0,2}

BitMap([0, 2])

In [ ]:
# engine.query_and([
#     engine.lookup("name","Alex"),
#     engine.lookup("theme","dark")
# ])
# {0}

BitMap([0])

In [ ]:
# engine.query_or([
#     engine.lookup("name","Alex"),
#     engine.lookup("country","India")
# ])

BitMap([0, 1, 2])

In [ ]:
# engine.query_not(
#     engine.lookup("country","USA")
# )
# {1,2}

BitMap([1, 2])

In [ ]:
# engine.query_in(
#     "country",
#     ["USA","India"]
# )
# {0,1,2}

BitMap([0, 1, 2])

In [ ]:
# engine.query_not_in(
#     "theme",
#     ["dark"]
# )
# {2}

BitMap([2])

### BSI

In [18]:
docs = [
    {"id": 0, "age": 5},
    {"id": 1, "age": 3},
    {"id": 2, "age": 6},
    {"id": 3, "age": 1},
    {"id": 4, "age": 7},
    {"id": 5, "age": 4},
    {"id": 6, "age": 2},
    {"id": 7, "age": 0}
]

| Doc | Age | Binary |
| --: | --: | :----: |
|   0 |   5 |   101  |
|   1 |   3 |   011  |
|   2 |   6 |   110  |
|   3 |   1 |   001  |
|   4 |   7 |   111  |
|   5 |   4 |   100  |
|   6 |   2 |   010  |
|   7 |   0 |   000  |


Look at the leftmost bit.Which document IDs have bit 2 = 1? 

0 2 4 5

similarly, Slice1 = BitMap([1,2,4,6])

and Slice0 = BitMap([0,1,3,4])

In [1]:
from datetime import datetime,date
import re

In [ ]:
class BSI:
  def __init__(self):
    self.slices=[]
    self.all_docs=BitMap()

  def ensure_slices(self,value):
    bits = max(1, value.bit_length())
    while len(self.slices)<bits:
      self.slices.append(BitMap())

  def add(self,doc_id,value):
    self.ensure_slices(value)
    self.all_docs.add(doc_id)
    for i in range(value.bit_length()):
      if ((value>>i)&1)==1:       #if the i'th bit is 1 we add the document to slice[i]
        self.slices[i].add(doc_id)

  def print_slices(self):
    for i, bitmap in enumerate(self.slices):
        print(f"Slice {i}: {list(bitmap)}")

  def equals(self,value):
    answer=self.all_docs.copy()
    for i in range(len(self.slices)):
      if ((value>>i)&1)==1:
        answer&=self.slices[i]
      else:
        zero_bitmap=self.all_docs-self.slices[i]
        answer&=zero_bitmap
    return answer

  def greater_than(self,value):
    equal=self.all_docs.copy()#initially all documents present here
    greater=BitMap()#initially empty

    for i in reversed(range(len(self.slices))):
      bit=(value>>i)&1
      if bit==1:
      #eg if ith  bit of value is 1; out of all documents only documents whose ith 1 can be tied and later be > value
      #who has ith bit 1 in this case? slice[i] so we intersect from that
        equal&=self.slices[i]
      else:#if i'th bit is zero
        #out of all documents
        #documents with i'th bit 1 automatically become greater
        #remaining all leftover docs must have ith bit 0
        greater|=equal&self.slices[i]
        zero_bitmap=self.all_docs-self.slices[i]
        equal&=zero_bitmap
    return greater

  def greater_than_equals(self,value):
    return self.greater_than(value)|self.equals(value)

  def less_than(self,value):
    return self.all_docs-(self.greater_than(value)|self.equals(value))

  def less_than_equals(self,value):
    return self.less_than(value)|self.equals(value)

  def get_value(self, doc_id): #returns the original value of a document
    value=0
    for i in range(len(self.slices)):
      if(self.slices[i].contains(doc_id)):
        value|=(1<<i)
    return value


In [33]:
bsi = BSI()

for doc in docs:
    bsi.add(doc["id"], doc["age"])


In [34]:
bsi.print_slices()

Slice 0: [0, 1, 3, 4]
Slice 1: [1, 2, 4, 6]
Slice 2: [0, 2, 4, 5]


In [35]:
print(bsi.greater_than(5))

BitMap([2, 4])


In [36]:
print(bsi.greater_than(2))

BitMap([0, 1, 2, 4, 5])


In [37]:
print(bsi.greater_than(7))

BitMap([])


In [38]:
print(bsi.less_than_equals(5))

BitMap([0, 1, 3, 5, 6, 7])


In [ ]:
class QueryNode:
        def __init__(self,operator=None,left=None,right=None,field=None,value=None,comparison=None):
          self.operator = operator
          self.left = left
          self.right = right
          self.field = field #condition=("age", ">", 25)
          self.value = value
          self.comparison = comparison

        def is_leaf(self):
          return self.field is not None


In [ ]:
from datetime import datetime,date

class SearchEngine:
    def __init__(self, ignored_fields=None):
        self.schema = {}
        self.index = {}#categorical values
        self.bsi_index={}#numerical values
        self.documents = {}
        self.all_docs=BitMap();
        self.next_doc_id = 0
        self.float_scale=1000
        self.ignored_fields = {
            field.lower()
            for field in (ignored_fields or set())
        }
        self.KEYWORDS={"NOT IN", "BETWEEN", "AND", "OR", "NOT", "IN"}
        self.VALID_OPERATORS = {"=","==","!=",">",">=","<","<=","IN","NOT IN","BETWEEN"}
        self.tokens = []
        self.current = 0
        self.field_present = {}

    # Determine field type
    def determine_type(self, value):
        if isinstance(value, bool):
            return "bool"
        elif isinstance(value, int):
            return "int"
        elif isinstance(value, float):
            return "float"
        elif isinstance(value, str):
            try:
                datetime.strptime(value, "%Y-%m-%d")
                return "date"
            except ValueError:
                return "categorical"
        return "unknown"

    # Build schema
    def build_schema(self, doc):
        for key, value in doc.items():
            if key.lower() in self.ignored_fields:
                continue
            if key in self.schema:
                continue
            field_type = self.determine_type(value)
            if field_type == "unknown":
                raise ValueError(f"Unsupported field type for {key}")
            if (field_type == "int" or field_type=="float") and value < 0:
                raise ValueError("Negative integers are not supported.")
            self.schema[key] = {
                "type": field_type,
                "bitmap": field_type in ("categorical", "bool"),
                "bsi": field_type in ("int", "float", "date"),
            }

    def encode_numeric(self, field, value):
        field_type = self.schema[field]["type"]
        if field_type == "date":
            return datetime.strptime(value, "%Y-%m-%d").toordinal()
        if field_type == "float":
            return int(round(value * self.float_scale))
        return value

    # Index one document
    def index_document(self, doc):
        # Update schema with any newly discovered fields
        self.build_schema(doc)

        #vaildation
        for field, value in doc.items():
            # Ignore fields that aren't indexed
            if field.lower() in self.ignored_fields:
                continue

            field_type = self.determine_type(value)

            if field_type=="unknown":
                raise ValueError(f"Unsupported type for field {field}")

            if field not in self.schema:
                raise ValueError(f"Unknown field {field}")

            if field_type != self.schema[field]["type"]:
                raise ValueError(f"{field} should be {self.schema[field]['type']}")
            if field_type in ("int", "float") and value < 0:
                raise ValueError("Negative integers are not supported.")
            if field not in self.field_present:
                self.field_present[field] = BitMap()
            self.field_present[field].add(doc_id)

        # Save original document
        doc_id = self.next_doc_id
        self.documents[doc_id] = doc
        self.all_docs.add(doc_id)

        #index
        for field, value in doc.items():
            if field.lower() in self.ignored_fields:
                continue
            #BSI
            if self.schema[field]["bsi"]:
                if field not in self.bsi_index:
                    self.bsi_index[field] = BSI()
                encoded_value = self.encode_numeric(field, value)
                self.bsi_index[field].add(doc_id, encoded_value)
            #bitmap
            if not self.schema[field]["bitmap"]:
                continue
            if field not in self.index:
                self.index[field] = {}
            if value not in self.index[field]:
                self.index[field][value] = BitMap()
            self.index[field][value].add(doc_id)
        self.next_doc_id += 1

    def validate_query_value(self, field, value):
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        expected = self.schema[field]["type"]
        actual = self.determine_type(value)
        if actual in ("int", "float") and value < 0:
            raise ValueError("Negative numeric values are not supported.")
        if actual != expected:
            raise ValueError(f"Field '{field}' expects {expected}, got {actual}")
    # Equality lookup
    def lookup(self, field, value):## returns bitmap
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        if field not in self.index:
            return BitMap()
        if value not in self.index[field]:
            return BitMap()
        return self.index[field][value]

    def query_and(self, results):
        if not results:
            return BitMap()
        answer = results[0].copy()
        for result in results[1:]:
            answer &= result
        return answer

    def query_or(self, results):
        if not results:
            return BitMap()
        answer = results[0].copy()
        for result in results[1:]:
            answer |= result
        return answer

    def query_not(self, result):
        return self.all_docs - result

    def query_in(self, field, values):
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        if self.schema[field]["bsi"]:
            if field not in self.bsi_index:
                return BitMap()
            results = []
            for value in values:
                self.validate_query_value(field, value)
                value = self.encode_numeric(field, value)
                results.append(self.bsi_index[field].equals(value))#BSI returns bitmap of matching documents
            return self.query_or(results)#which is why query or works here
        else:
            results = []
            for value in values:
                results.append(self.lookup(field, value))
            return self.query_or(results)

    def query_not_in(self, field, values):#works for both categorical and numeric
        result = self.query_in(field, values)
        return self.query_not(result)

    # batch ingest
    def index_documents(self, docs):
        for doc in docs:
            self.index_document(doc)

    def greater_than(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].greater_than(value)

    def greater_than_equals(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].greater_than_equals(value)

    def less_than(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].less_than(value)

    def less_than_equals(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].less_than_equals(value)

    def equals_numeric(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].equals(value)

    def between(self, field, low, high):
        if low > high:
            raise ValueError("low must be <= high")
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        self.validate_query_value(field, low)
        self.validate_query_value(field, high)
        low = self.encode_numeric(field, low)
        high = self.encode_numeric(field, high)
        return (self.bsi_index[field].greater_than_equals(low) & self.bsi_index[field].less_than_equals(high))

    def get_value(self, field, doc_id):
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.bsi_index[field].get_value(doc_id)
        if self.schema[field]["type"] == "date":
            return date.fromordinal(value).strftime("%Y-%m-%d")
        elif self.schema[field]["type"] == "float":
            return value / self.float_scale
        return value

## QUERY PARSING
    def tokenize(self, query):
        pattern = r'"[^"]*"|\'[^\']*\'|\bNOT\s+IN\b|\bBETWEEN\b|\bAND\b|\bOR\b|\bNOT\b|\bIN\b|>=|<=|!=|==|=|>|<|\(|\)|,|[A-Za-z0-9_.-]+'
        # Detect unterminated quotes before tokenizing -- check each quote
        # style independently, since one could be balanced while the other isn't.
        if query.count('"') % 2 != 0:
            raise ValueError(f"Unterminated double-quoted string in query: {query!r}")
        if query.count("'") % 2 != 0:
            raise ValueError(f"Unterminated single-quoted string in query: {query!r}")
        pos = 0
        tokens = []
        for match in re.finditer(pattern, query, flags=re.IGNORECASE):
            # Catch characters between matches that the pattern doesn't cover
            # (e.g. *, %, $) instead of silently skipping them.
            gap = query[pos:match.start()]
            if gap.strip():
                raise ValueError(f"Unrecognized character(s) {gap.strip()!r} in query: {query!r}")
            pos = match.end()
            token = match.group()
            token = token.strip()
            if (token.startswith('"') and token.endswith('"')) or \
               (token.startswith("'") and token.endswith("'")):
                token = token[1:-1]
            else:
                # Collapse internal whitespace (handles "NOT   IN" -> "NOT IN")
                token = re.sub(r'\s+', ' ', token)
                # Normalize keyword casing so downstream comparisons are reliable
                if token.upper() in self.KEYWORDS:
                    token = token.upper()
            tokens.append(token)
        # Check for trailing unrecognized characters after the last match
        trailing = query[pos:]
        if trailing.strip():
            raise ValueError(f"Unrecognized character(s) {trailing.strip()!r} in query: {query!r}")
        return tokens

    def peek(self):
        if self.current >= len(self.tokens):
            return None
        return self.tokens[self.current]
    def consume(self):
        token = self.peek()
        if token is None:
            raise ValueError("Unexpected end of query")
        self.current += 1
        return token

    def parse(self, query):
        self.tokens = self.tokenize(query)
        self.current = 0
        root = self.parse_expression()
        if self.peek() is not None:
            raise ValueError(f"Unexpected token '{self.peek()}'")
        return root

    def parse_expression(self):
        return self.parse_or()

    def parse_or(self):
        node = self.parse_and()#parse everything on the left side fist as and has higher presedence
        while self.peek() == "OR":
            self.consume()
            right = self.parse_and()#parse everything on the right side
            node = QueryNode(operator="OR",left=node,right=right)
        return node

    def parse_and(self):
        node = self.parse_not()#parse everything on the left side fist as and has higher presedence
        while self.peek() == "AND":
            self.consume()
            right = self.parse_not()#parse everything on the right side
            node = QueryNode(operator="AND",left=node,right=right)
        return node

    def parse_not(self):
        if self.peek() == "NOT":
          self.consume()
          return QueryNode(operator="NOT",left=self.parse_not())
        return self.parse_primary()

    def parse_primary(self):
        if self.peek() == "(":
          self.consume()
          node = self.parse_expression()
          if self.peek() != ")":
            raise ValueError("Expected ')'")
          self.consume()
          return node
        return self.parse_condition()

    def parse_condition(self):
        field = self.consume()
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        operator = self.consume()
        if operator not in self.VALID_OPERATORS:
            raise ValueError(f"Unknown operator '{operator}'")
        # BETWEEN
        if operator == "BETWEEN":
          low = self.consume()
          if self.consume() != "AND":
            raise ValueError("Expected AND inside BETWEEN")
          high = self.consume()
          low = self.convert_value(field, low)
          high = self.convert_value(field, high)
          return QueryNode(field=field,comparison="BETWEEN",value=(low, high))
        # IN / NOT IN
        elif operator in ("IN", "NOT IN"):
          if self.consume() != "(":
            raise ValueError("Expected '('")
          values = []
          while True:
            values.append(self.convert_value(field, self.consume()))
            token = self.consume()
            if token == ")":
              break
            if token != ",":
              raise ValueError("Expected ',' or ')'")
          return QueryNode(field=field,comparison=operator,value=values)
        # Normal comparison
        else:
          value = self.consume()
          value = self.convert_value(field, value)
          return QueryNode(field=field,comparison=operator,value=value)

    def convert_value(self, field, value):
        field_type = self.schema[field]["type"]
        if field_type == "int":
          return int(value)
        elif field_type == "float":
          return float(value)
        elif field_type == "bool":
          if value.lower() == "true":
            return True
          if value.lower() == "false":
            return False
          raise ValueError("Boolean must be true or false")
        return value

    def evaluate(self, node):
        if node.is_leaf():
          return self.execute_condition(node.field,node.comparison,node.value)
        if node.operator == "AND":
          return self.query_and([self.evaluate(node.left),self.evaluate(node.right)])
        if node.operator == "OR":
          return self.query_or([self.evaluate(node.left),self.evaluate(node.right)])
        if node.operator == "NOT":
          return self.query_not(self.evaluate(node.left))
        raise ValueError("Unknown node")

    def search(self, query):
        tree = self.parse(query)
        bitmap = self.evaluate(tree)
        return self.fetch(bitmap)


    def execute_condition(self, field, operator, value):
        # Numeric / Date fields
        if self.schema[field]["bsi"]:
            if operator in ("=","=="):
                return self.equals_numeric(field, value)
            elif operator == "!=":
                return self.query_not(self.equals_numeric(field, value))
            elif operator == ">":
                return self.greater_than(field, value)
            elif operator == ">=":
                return self.greater_than_equals(field, value)
            elif operator == "<":
                return self.less_than(field, value)
            elif operator == "<=":
                return self.less_than_equals(field, value)
            elif operator == "IN":
                return self.query_in(field, value)
            elif operator == "NOT IN":
                return self.query_not_in(field, value)
            elif operator == "BETWEEN":
                low, high = value
                return self.between(field, low, high)
            else:
                raise ValueError(f"{operator} is not supported for numerical fields")
        # Categorical fields
        else:
            if operator in ("=","=="):
                return self.lookup(field, value)
            elif operator == "!=":
                self.validate_query_value(field, value)
                return self.query_not(self.lookup(field, value))
            elif operator == "IN":
                return self.query_in(field, value)
            elif operator == "NOT IN":
                return self.query_not_in(field, value)
            else:
                raise ValueError(f"{operator} is not supported for categorical fields")

    def fetch(self, bitmap):#converts bitmap into actual documents as bitmaps return the doc_id
        docs=[]
        for doc_id in bitmap:#bitmap is the result of a query -->works on bsi too as bsi returns bitmap of doc_ids
            docs.append(self.documents[doc_id])
        return docs


    #cardinality Analysis
    def analyse_cardinality(self):
        for field in self.index:
            self.schema[field]["cardinality"] = len(self.index[field])
        return self.schema

    # def disable_high_cardinality(self,threshold):
    #     self.analyse_cardinality()
    #     for field in self.schema:
    #         if self.schema[field]["cardinality"]>threshold:
    #             self.schema[field]["indexed"]=False

### Query Parser